In [12]:
import math
from platform import system

from dotenv import load_dotenv



load_dotenv()

True

In [7]:
from langchain_core.tools import tool
@tool
def square_root(x: float) -> float:
    """calculate the square root of a number"""
    return math.sqrt(x)

In [8]:
from pydantic import BaseModel,Field
from typing import Literal

class WeatherInput(BaseModel):
    """weather input"""
    location: str = Field( description="location of the weather report"),
    units: Literal["celsius","fahrenheit"] = Field(
        default="celsius",
        description= "units of the weather report"
    )
    include_forecast:bool = Field(
        default=False,
        description=" Include 5-day forecast"
    )
@tool(args_schema=WeatherInput)
def get_weather(location:str,units:str = "celsius",include_forecast:bool = False)->str:
    """Get weather report"""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degree {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days :Sunny"
    return result

In [9]:
from langchain.agents import create_agent
agent = create_agent(
    model="deepseek-chat",
    tools = [square_root,get_weather],
    system_prompt = "你是一个智能助手，你使用工具来解决用户问题。"
)

In [10]:
from langchain.messages import HumanMessage
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="467的平方根是多少?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="北京和杭州接下来几天天气如何?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

D:\data\AI agent\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='location of the weather report'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


21.61018278497431467 的平方根是 **约 21.6102**（精确值为 21.61018278497431）。I'll check the weather for both cities for you.Current weather in 北京: 22 degree C
Next 5 days :SunnyCurrent weather in 杭州: 22 degree C
Next 5 days :Sunny北京和杭州接下来几天的天气情况如下：

| 城市 | 当前气温 | 未来5天 |
|------|---------|---------|
| 北京 | 22°C | 晴 |
| 杭州 | 22°C | 晴 |

两座城市目前气温都是 22°C，且未来 5 天都是晴朗天气。天气不错，适合外出活动，不过早晚可能有温差，建议出门时备一件外套，白天注意防晒。

In [13]:
from langchain_tavily import TavilySearch
search_tool = TavilySearch(
    max_results=5,
    topic = "general"
        # include_answer=False,
    # include_raw_content=False,
    # include_images=False,
    # include_image_descriptions=False,
    # search_depth="basic",
    # time_range="day",
    # include_domains=None,
    # exclude_domains=None
)

In [14]:
search_tool.invoke("南邮毕业生平均年薪多少？")

{'query': '南邮毕业生平均年薪多少？',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.163.com/dy/article/L5DMHNQF05563JRK.html',
   'title': '98%落实率+华为批量抢人！南京邮电大学这份就业成绩单太硬核了',
   'content': 'A：南邮毕业生的薪资极具竞争力。普通本科毕业生起薪普遍在8000-12000元/月，而集成电路等热门专业的应届生平均年薪更是突破35万元，位居江苏省属高校前列',
   'score': 0.9053309,
   'raw_content': None,
   'id': '899e79-00'},
  {'url': 'https://www.eefocus.com/article/1854585.html',
   'title': '南邮2024年就业报告，华为202人！ - 与非网',
   'content': '在互联网领域，阿里巴巴、腾讯、百度等知名大厂也向南邮毕业生抛出橄榄枝，为其提供互联网技术研发、产品设计、运营管理等关键岗位，助力毕业生在数字经济浪潮中施展才华。除企业外，不少毕业生选择进入政府机构，参与通信政策制定、技术标准规范、行业监管等工作；或是投身科研院所，协助完成国家重点研究项目，推动学科前沿研究成果转化。中国邮政、中国电科等央企和国企，也凭借稳定的工作环境，吸引了众多南邮学子加入。\n\n在就业质量上，南邮表现出色。2023 届本科毕业生的年终就业去向落实率达到了 94.07%，研究生的就业去向落实率更是在 99% 以上，而 2024 届本科毕业生的深造率超过了 30%。在就业行业认可度上，南邮毕业生在信息传输、软件和信息技术服务业备受追捧，2023 届超过半数的本科生和研究生都流向这一领域。这得益于南邮在信息通信方面深厚的历史底蕴和强大的学科实力，其通信工程、计算机科学与技术等专业教学水平和科研成果突出，让学生在校就能接触行业前沿知识与技术，练就过硬专业技能。就业薪资方面，研究生年薪普遍超过 30 万元，本科生薪资也处于较高水平。 [...] 电子硬件助手\n元器件查询\n\n热搜\n\

In [16]:
agent = create_agent(
    model="deepseek-chat",
    tools=[search_tool],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题"
)


In [19]:
response = agent.invoke(
    {"messages":[HumanMessage(content="南邮是不是吊打一些211？")]}
)
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

南邮是不是吊打一些211？
================================== Ai Message ==================================

这是个挺有意思的问题。我理解你说的"南邮"应该是指**南京邮电大学**。不过"吊打一些211"这个说法比较主观,涉及具体指标和哪些211的比较,我想先确认一下你的意思,同时帮你查点实际数据。

先说个背景:南邮是"双一流"建设高校,但**不是211**。它最强的领域是通信、电子信息、计算机类,尤其在通信行业口碑很硬,和中兴、华为、三大运营商关系密切。

我搜一下最新的排名、录取分数、学科评估等数据来给你更靠谱的对比。
Tool Calls:
  tavily_search (call_00_9xh5xSsPFuSIcRvf775k0282)
 Call ID: call_00_9xh5xSsPFuSIcRvf775k0282
  Args:
    query: 南京邮电大学 2024 录取分数线 对比 211 排名
    search_depth: advanced
  tavily_search (call_01_4Ds48hPKzws9IVd2YSCs4909)
 Call ID: call_01_4Ds48hPKzws9IVd2YSCs4909
  Args:
    query: 南京邮电大学 学科评估 通信工程 电子科学与技术 就业
    search_depth: advanced
================================= Tool Message =================================
Name: tavily_search

{"query": "南京邮电大学 2024 录取分数线 对比 211 排名", "follow_up_questions": null, "answer": null, "images": [], "results": [{"title": "南京邮电大学2024年录取分数线：文科492分，理科450分", "url"

In [20]:
tavily = TavilySearch(
    max_results=5,
    topic = "general"
)
@tool
def web_search(query:str):
    """Search web page"""
    return  tavily.invoke(query)

In [21]:
from pydantic import BaseModel,Field
class Reference(BaseModel):
    title:str = Field(description="title of the reference"),
    url:str = Field(description="url of the reference")
class AnswerInfo (BaseModel):
    answer :str = Field(description="answer to the question")
    reference:list[Reference] = Field(description="reference to the question")

In [25]:
agent = create_agent(
    model="deepseek-chat",
    tools=[web_search],
    system_prompt="你是一个智能助手，你使用工具来解决用户问题。",
    response_format=AnswerInfo
)
response = agent.invoke(
     {"messages": [HumanMessage(content="南邮和西南交通大学比能吊打吗")]},
)
print(response['structured_response'])
for message in response['messages']:
    message.pretty_print()

D:\data\AI agent\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='title of the reference'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


answer='结论先说：**谈不上"吊打"，两者各有强项，属于不同赛道的学校，谁强取决于你看哪个维度。**\n\n## 一、学校层次与身份\n\n| 维度 | 南京邮电大学 | 西南交通大学 |\n|---|---|---|\n| 身份 | 双一流（非211、非985） | 211、双一流、985工程优势学科创新平台 |\n| 隶属 | 江苏省属，工信部/国家邮政局共建 | 教育部直属 |\n| 综合排名（软科2024） | 主榜第85名 | 约第59名（在各类综合榜单中通常领先南邮10~30名） |\n\n单看"帽子"和综合排名，**西南交大更占优**（211+部属+综合排名更高），南邮是双非（虽然入了双一流）。所以从整体招牌来说，南邮"吊打"西南交大是不成立的，反而略处下风。\n\n## 二、学科实力对比（第四轮学科评估）\n\n**西南交通大学：**\n- A+：交通运输工程（全国第一档，与东南、北交并列）\n- A-：土木工程\n- B+：机械、电气、信息与通信工程、计算机、管理科学与工程、工商管理、马克思主义理论\n- 拥有轨道交通国家实验室（全国仅6所高校有），工科底子极厚\n\n**南京邮电大学：**\n- 无A类学科\n- B+：光学工程、电子科学与技术、信息与通信工程\n- B：软件工程；B-：计算机科学与技术\n- 双一流建设学科：电子科学与技术\n\n从学科评估看，**西南交大A类学科层次更高、上榜学科更多、门类更全**；南邮没有A类，但在电子信息这个细分赛道上非常集中、特色鲜明。\n\n## 三、细分领域谁更强\n\n**信息与通信工程（两者都有）**：软科2024中国最好学科排名中，南邮第13名（前7%），西南交大第21名（前12%）——**南邮在这个点上反超西南交大**。南邮通信工程软科专业排名全国第6，通信工程世界排名全球第26，是它的看家本领。\n\n**其他方向**：土木、机械、电气、交通、测绘、轨道交通等，西南交大全面领先，南邮基本没有可比性。\n\n## 四、就业与录取分\n\n**就业**：南邮在通信/互联网行业认可度极高，2024届华为录用202人，中兴、小米等大厂也大量招聘，三大运营商录用近800人，研究生年薪普遍30万+。**在ICT就业市场上，南邮的认可度不输甚至优于很多211**，这正是它最大的"性价比"卖点。而西